# 01 · Recolección de datos con la API REST de SpaceX

**Objetivo:** construir el dataset de lanzamientos del Falcon 9 a partir de la API pública de SpaceX
(`https://api.spacexdata.com/v4`).

**Flujo**

1. Descargar el histórico de lanzamientos. Igual que en el laboratorio de IBM, se usa la copia estática
   `API_call_spacex_api.json` para que los resultados sean reproducibles (la API en vivo cambia con el tiempo).
2. Los lanzamientos solo traen identificadores (`rocket`, `launchpad`, `payloads`, `cores`). Se resuelven
   consultando los endpoints `/rockets`, `/launchpads`, `/payloads` y `/cores`.
3. Filtrar misiones con un único núcleo y una única carga útil, y fechas hasta el 13-nov-2020.
4. Conservar solo el **Falcon 9**, imputar la masa de carga faltante con la media y guardar
   `data/dataset_part_1.csv`.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [2]:
import datetime
import json

import numpy as np
import requests

pd.set_option("display.max_columns", None)

SPACEX_API = "https://api.spacexdata.com/v4"
STATIC_JSON_URL = f"{IBM}/datasets/API_call_spacex_api.json"
FALLBACK_CSV_URL = f"{IBM}/datasets/dataset_part_1.csv"

session = requests.Session()
session.headers["User-Agent"] = "ibm-capstone-spacex/1.0"

## 1. Petición de prueba a la API en vivo

Se comprueba que la API responde (código 200). El análisis usa la copia estática, pero esta celda demuestra
el uso de `requests` contra el endpoint real.

In [3]:
try:
    live = session.get(f"{SPACEX_API}/launches/past", timeout=30)
    print("Código de estado de la API en vivo:", live.status_code)
except requests.RequestException as exc:
    print("La API en vivo no respondió:", exc)

Código de estado de la API en vivo: 525


## 2. Descarga del histórico de lanzamientos (copia estática del curso)

In [4]:
response = session.get(STATIC_JSON_URL, timeout=60)
response.raise_for_status()
data = pd.json_normalize(response.json())
print(data.shape)
data.head()

(107, 42)


,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'mer...",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight'...",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-fa...,https://en.wikipedia.org/wiki/DemoSat,NaN
1,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Successful first stage burn and transition to ...,[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'har...",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight'...",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-roc...,https://en.wikipedia.org/wiki/DemoSat,NaN
2,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision betwe...,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006e...",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'resi...",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight'...",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1...,https://en.wikipedia.org/wiki/Trailblazer_(sat...,NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,Ratsat was carried to orbit on the first succe...,[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight'...",5eb87cdbffd86e000604b32d,False,False,False,[],https://images2.imgbox.com/e9/c9/T8CfiSYb_o.png,https://images2.imgbox.com/e0/a7/FNjvKlXW_o.png,NaN,NaN,NaN,NaN,[],[],NaN,https://www.youtube.com/watch?v=dLQ2tZEH6G0,dLQ2tZEH6G0,https://en.wikipedia.org/wiki/Ratsat,https://en.wikipedia.org/wiki/Ratsat,NaN
4,NaN,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,True,NaN,[],[],[],[5eb0e4b7b6c3bb0006eeb1e6],5e9e4502f5090995de566f86,True,[],5,RazakSat,2009-07-13T03:35:00.000Z,1247456100,2009-07-13T15:35:00+12:00,hour,False,"[{'core': '5e9e289ef359184f103b2627', 'flight'...",5eb87cdcffd86e000604b32e,False,False,False,[],https://images2.imgbox.com/a7/ba/NBZSw3Ho_o.png,https://images2.imgbox.com/8d/fc/0qdZMWWx_o.png,NaN,NaN,NaN,NaN,[],[],http://www.spacex.com/press/2012/12/19/spacexs...,https://www.youtube.com/watch?v=yTaIDooc8Og,yTaIDooc8Og,http://www.spacex.com/news/2013/02/12/falcon-1...,https://en.wikipedia.org/wiki/RazakSAT,NaN


## 3. Selección de columnas y filtros

* Solo misiones con **un** núcleo (`cores`) y **una** carga útil (`payloads`): las misiones con varios
  boosters (Falcon Heavy) o varias cargas complican el análisis.
* Fecha de corte: 13-nov-2020, igual que el laboratorio.

In [5]:
data = data[["rocket", "payloads", "launchpad", "cores", "flight_number", "date_utc"]]
data = data[data["cores"].map(len) == 1]
data = data[data["payloads"].map(len) == 1]
data["cores"] = data["cores"].map(lambda x: x[0])
data["payloads"] = data["payloads"].map(lambda x: x[0])
data["date"] = pd.to_datetime(data["date_utc"]).dt.date
data = data[data["date"] <= datetime.date(2020, 11, 13)]
print(data.shape)

(94, 7)


## 4. Enriquecimiento con los endpoints de la API

Funciones auxiliares que, a partir de cada identificador, consultan la API y extraen los campos que
necesitamos. Se usa una caché en memoria para no repetir la misma petición.

In [6]:
_cache: dict[str, dict] = {}


def get_json(endpoint: str, item_id: str) -> dict:
    key = f"{endpoint}/{item_id}"
    if key not in _cache:
        r = session.get(f"{SPACEX_API}/{key}", timeout=30)
        r.raise_for_status()
        _cache[key] = r.json()
    return _cache[key]


BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []


def getBoosterVersion(df):
    for rocket_id in df["rocket"]:
        BoosterVersion.append(get_json("rockets", rocket_id)["name"])


def getLaunchSite(df):
    for pad_id in df["launchpad"]:
        pad = get_json("launchpads", pad_id)
        Longitude.append(pad["longitude"])
        Latitude.append(pad["latitude"])
        LaunchSite.append(pad["name"])


def getPayloadData(df):
    for payload_id in df["payloads"]:
        payload = get_json("payloads", payload_id)
        PayloadMass.append(payload["mass_kg"])
        Orbit.append(payload["orbit"])


def getCoreData(df):
    for core in df["cores"]:
        if core["core"] is not None:
            info = get_json("cores", core["core"])
            Block.append(info["block"])
            ReusedCount.append(info["reuse_count"])
            Serial.append(info["serial"])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(f"{core['landing_success']} {core['landing_type']}")
        Flights.append(core["flight"])
        GridFins.append(core["gridfins"])
        Reused.append(core["reused"])
        Legs.append(core["legs"])
        LandingPad.append(core["landpad"])

In [7]:
try:
    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)
    api_ok = True
except requests.RequestException as exc:
    api_ok = False
    print("No se pudo completar el enriquecimiento con la API:", exc)

No se pudo completar el enriquecimiento con la API: 525 Server Error: <none> for url: https://api.spacexdata.com/v4/rockets/5e9d0d95eda69955f709d1eb


## 5. Construcción del DataFrame

In [8]:
if api_ok:
    launch_dict = {
        "FlightNumber": list(data["flight_number"]),
        "Date": list(data["date"]),
        "BoosterVersion": BoosterVersion,
        "PayloadMass": PayloadMass,
        "Orbit": Orbit,
        "LaunchSite": LaunchSite,
        "Outcome": Outcome,
        "Flights": Flights,
        "GridFins": GridFins,
        "Reused": Reused,
        "Legs": Legs,
        "LandingPad": LandingPad,
        "Block": Block,
        "ReusedCount": ReusedCount,
        "Serial": Serial,
        "Longitude": Longitude,
        "Latitude": Latitude,
    }
    launch_df = pd.DataFrame(launch_dict)
    source = "SpaceX REST API v4"
else:
    # Respaldo documentado: el dataset_part_1.csv que publica IBM es el resultado de este mismo proceso.
    print("Usando el respaldo oficial del curso:", FALLBACK_CSV_URL)
    launch_df = pd.read_csv(FALLBACK_CSV_URL)
    source = "IBM dataset_part_1.csv (respaldo)"
launch_df.head()

Usando el respaldo oficial del curso: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 6. Filtrar solo Falcon 9 y renumerar los vuelos

In [9]:
data_falcon9 = launch_df[launch_df["BoosterVersion"] != "Falcon 1"].copy()
data_falcon9["FlightNumber"] = list(range(1, data_falcon9.shape[0] + 1))
print("Lanzamientos Falcon 9:", len(data_falcon9))
data_falcon9.isnull().sum()

Lanzamientos Falcon 9: 90


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

## 7. Tratamiento de valores faltantes

`PayloadMass` tiene valores nulos: se reemplazan por la **media**. `LandingPad` conserva sus nulos, porque
significan "no se usó plataforma de aterrizaje" y esa ausencia es información.

In [10]:
payload_mean = data_falcon9["PayloadMass"].astype(float).mean()
data_falcon9["PayloadMass"] = data_falcon9["PayloadMass"].astype(float).fillna(payload_mean)
print(f"Media usada para imputar PayloadMass: {payload_mean:,.2f} kg")
data_falcon9.isnull().sum()

Media usada para imputar PayloadMass: 6,104.96 kg


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

In [11]:
data_falcon9.to_csv(DATA_DIR / "dataset_part_1.csv", index=False)

summary = {
    "source": source,
    "rows": int(len(data_falcon9)),
    "columns": list(data_falcon9.columns),
    "payload_mean_imputed_kg": round(float(payload_mean), 2),
}
(RESULTS_DIR / "data_collection_api.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

{'source': 'IBM dataset_part_1.csv (respaldo)',
 'rows': 90,
 'columns': ['FlightNumber',
  'Date',
  'BoosterVersion',
  'PayloadMass',
  'Orbit',
  'LaunchSite',
  'Outcome',
  'Flights',
  'GridFins',
  'Reused',
  'Legs',
  'LandingPad',
  'Block',
  'ReusedCount',
  'Serial',
  'Longitude',
  'Latitude'],
 'payload_mean_imputed_kg': 6104.96}

## Conclusión

Se obtuvo un dataset limpio de lanzamientos Falcon 9 con 17 variables (versión del booster, masa de carga,
órbita, sitio, resultado del aterrizaje, reutilización, patas, aletas de rejilla, etc.), listo para el
*data wrangling* del notebook 03.